In [ ]:
%%capture
!pip install -U transformers accelerate safetensors sentencepiece tokenizers datasets pandas tqdm huggingface_hub fasttext-wheel

In [ ]:
# ── Build ALL init arms in ONE run, derived from the existing v5 SALT artifact ──
# v5 is the expensive source (anchor mining + translation + fastText projection, nb01).
# This notebook DERIVES every comparison arm from it by copying v5's model dir and
# swapping tensors — no re-mining, no retraining. It does NOT rebuild v5 (run nb01 for
# that) and does NOT include freeze-align (nb14 is a separate TRAINING stage).
#
# Arms produced:
#   A  trung_salt_globalmap_freqbias  (SALT method, the reference — rebuild via nb01 first)
#   B  trung_random_meannorm              (scale-matched random control: isolates emb semantics)
#   C  trung_naive_random_zerobias         (naive vocab-swap CPT: random emb+dec, zero bias)
#   trung_salt_decscale05 / _decscale10 / _dectied / _decpertoken (decoder-method variants)
import sys, json, shutil, glob
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')
import importlib
import salt3_common as sc; importlib.reload(sc)
import salt3_decoder_variants as sdv; importlib.reload(sdv)
from salt3_common import (configure_environment, set_seed, ensure_dir,
                          fit_embedding_to_decoder_map, load_tokenizer_no_remote_code)
configure_environment(); set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SOURCE_MODEL_ID = 'chandar-lab/NeoBERT'
V5_NAME  = 'trung_salt_globalmap_freqbias'
V5_DIR   = PROJECT_ROOT / 'init' / V5_NAME
V5_MODEL = V5_DIR / 'model'
VARIANT_PREFIX = 'trung_salt'

# ── build toggles ──
BUILD_PERTOKEN       = True   # per-token decoder arm (needs ~7GB fastText + ViDeBERTa donor)
DECODER_WEIGHT_SCALE = 0.1    # v5's decoder scale (shared by random control + scale ref)

assert V5_DIR.exists(), f'v5 source init not found: {V5_DIR} (run nb01 first)'
print('source v5 :', V5_DIR)
print('device    :', DEVICE, '| torch', torch.__version__)

In [ ]:
# ── Load shared tensors ONCE ──
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

# NeoBERT source tensors straight from hub safetensors (no xformers check_imports)
neo_st  = load_file(hf_hub_download(SOURCE_MODEL_ID, 'model.safetensors'))
neo_emb = neo_st['model.encoder.weight'].float()
neo_dec = neo_st['decoder.weight'].float()

# v5 artifact tensors (the source everything derives from)
v5_st   = load_file(str(V5_MODEL / 'model.safetensors'))
emb_v5  = v5_st['model.encoder.weight'].float()
dec_v5  = v5_st['decoder.weight'].float()          # = E_salt @ M * V5_SCALE
bias_v5 = v5_st['decoder.bias'].float()            # Vietnamese unigram log-freq prior
V, HID  = emb_v5.shape

# tokenizers -> special-token role map (target<-source), shared by the random arms
src_tok = load_tokenizer_no_remote_code(SOURCE_MODEL_ID, V5_DIR / 'neobert_tokenizer_local')
tgt_tok = AutoTokenizer.from_pretrained(V5_MODEL, trust_remote_code=True)
src_vocab, tgt_vocab = src_tok.get_vocab(), tgt_tok.get_vocab()
special_pairs = sdv.special_token_pairs(tgt_tok, src_tok, tgt_vocab, src_vocab)

# NeoBERT init ranges from the v5 model config — used by the naive from-scratch arm
v5_cfg  = json.loads((V5_MODEL / 'config.json').read_text(encoding='utf-8'))
EMB_INIT_RANGE = float(v5_cfg.get('embedding_init_range', 0.02))
DEC_INIT_RANGE = float(v5_cfg.get('decoder_init_range', v5_cfg.get('embedding_init_range', 0.02)))
salt_cfg = json.loads((V5_DIR / 'salt_config.json').read_text(encoding='utf-8'))
V5_SCALE = float(salt_cfg.get('decoder_weight_scale', 0.1) or 0.1)

assert V == len(tgt_tok), f'v5 emb rows {V} != tokenizer {len(tgt_tok)}'
print(f'vocab={V} hidden={HID} | specials={len(special_pairs)}')
print(f'NeoBERT init ranges : embedding={EMB_INIT_RANGE}  decoder={DEC_INIT_RANGE}')
print(f'v5 decoder scale={V5_SCALE} | freq-bias norm={bias_v5.norm():.1f} '
      f'range=[{bias_v5.min():.2f},{bias_v5.max():.2f}]')

In [ ]:
# ── arm registry + build helper ──
ARMS_BUILT = []  # (label, init_dir) for the final health gate

def arm_dir(name):
    return ensure_dir(PROJECT_ROOT / 'init' / name)

def build(label, init_name, tensor_updates, cfg_updates):
    d = PROJECT_ROOT / 'init' / init_name
    sdv.write_artifact(V5_DIR, d, tensor_updates, {'init_name': init_name, **cfg_updates})
    ARMS_BUILT.append((label, d))
    return d

In [ ]:
# ── B. Random control: trained body + scale-matched random emb + mapped decoder + freq bias.
#      Differs from v5 ONLY in embedding semantics (random directions at NeoBERT mean norm). ──
emb_to_dec, resid = fit_embedding_to_decoder_map(neo_emb, neo_dec)
print(f'global emb->dec residual: {resid:.3f}')

emb_B = sdv.random_embedding_matrix(V, HID, neo_emb, special_pairs, tgt_tok.pad_token_id,
                                    mode='neobert_meannorm', seed=42)
dec_B = (emb_B @ emb_to_dec) * DECODER_WEIGHT_SCALE
build('B random-meannorm', 'trung_random_meannorm',
      {'model.encoder.weight': emb_B, 'decoder.weight': dec_B, 'decoder.bias': bias_v5.clone()},
      {'type': 'random_baseline_control', 'embedding_init': 'random_neobert_meannorm',
       'decoder_init': 'global_emb_to_decoder_map', 'decoder_weight_scale': DECODER_WEIGHT_SCALE,
       'decoder_bias_init': 'copied_from_v5_vietnamese_unigram_logfreq',
       'anchor_pairs': 0, 'projection_stats': None})
print(f'  emb mean-norm {emb_B.norm(dim=1).mean():.4f} | dec mean row-norm {dec_B.norm(dim=1).mean():.4f}')

In [ ]:
# ── C. Naive CPT baseline: trained body + fully RANDOM new vocab rows (what an engineer gets
#      resizing the head for a new tokenizer): N(0,emb_range) emb + N(0,dec_range) INDEPENDENT
#      decoder + ZERO bias. No SALT projection, no emb->dec map, no freq prior. ──
emb_C = sdv.random_embedding_matrix(V, HID, neo_emb, special_pairs, tgt_tok.pad_token_id,
                                    mode='scratch_init_range', seed=42, init_range=EMB_INIT_RANGE)
dec_C = sdv.random_decoder_matrix(V, HID, DEC_INIT_RANGE, special_pairs, neo_dec, seed=43)

build('C naive (zero-bias)', 'trung_naive_random_zerobias',
      {'model.encoder.weight': emb_C, 'decoder.weight': dec_C, 'decoder.bias': torch.zeros(V)},
      {'type': 'naive_cpt_reference', 'embedding_init': f'scratch_N0_{EMB_INIT_RANGE}',
       'decoder_init': f'random_N0_{DEC_INIT_RANGE}', 'decoder_bias_init': 'zero',
       'anchor_pairs': 0, 'projection_stats': None})

print(f'  naive emb mean-norm {emb_C.norm(dim=1).mean():.4f} | dec mean row-norm {dec_C.norm(dim=1).mean():.4f}')

In [ ]:
# ── Decoder-construction variants (share v5's encoder + bias; only decoder.weight changes) ──
for arm, scale in (('decscale05', 0.5), ('decscale10', 1.0)):
    w = dec_v5 * (scale / V5_SCALE)              # dec_v5 already has V5_SCALE baked in
    name = f'{VARIANT_PREFIX}_{arm}'
    sdv.write_variant(V5_DIR, arm_dir(name), w,
                      {'init_name': name, 'decoder_init': 'global_emb_to_decoder_map',
                       'decoder_weight_scale': scale})
    ARMS_BUILT.append((arm, PROJECT_ROOT / 'init' / name))

name = f'{VARIANT_PREFIX}_dectied'
sdv.write_variant(V5_DIR, arm_dir(name), emb_v5.clone(),
                  {'init_name': name, 'decoder_init': 'tied_to_embeddings', 'decoder_weight_scale': None})
ARMS_BUILT.append(('dectied', PROJECT_ROOT / 'init' / name))
print('scale + tied variants written')

In [ ]:
# ── Per-token decoder arm (SALT Eq.3 local maps, NeoBERT decoder rows as targets) ──
if BUILD_PERTOKEN:
    import fasttext
    from transformers import AutoModel

    videberta = AutoModel.from_pretrained('Fsoft-AIC/videberta-base')
    vi_emb_full = sc.extract_embedding_weight(videberta).float().cpu(); del videberta

    full_tok_json = hf_hub_download('Fsoft-AIC/videberta-base', 'tokenizer.json')
    target_vocab, new_to_old = sdv.rebuild_vocab_maps(V5_DIR / 'pruned_videberta_tokenizer', full_tok_json)
    anchor_map = sdv.read_anchor_map(V5_DIR)
    specials = sdv.special_token_pairs(tgt_tok, src_tok, target_vocab, src_vocab)
    print(f'vocab={len(target_vocab)} anchors={len(anchor_map)} specials={len(specials)}')
    assert len(target_vocab) == V, 'pruned vocab != v5 embedding rows'
    assert len(anchor_map) > 3000, 'anchor CSVs incomplete — check v5 dir'

    FT_BIN = V5_DIR / 'cc.vi.300.bin'
    if not FT_BIN.exists():
        import gzip, urllib.request
        print('fastText bin missing — downloading ~7GB...')
        urllib.request.urlretrieve(
            'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.vi.300.bin.gz', str(FT_BIN) + '.gz')
        with gzip.open(str(FT_BIN) + '.gz', 'rb') as fi, open(FT_BIN, 'wb') as fo:
            shutil.copyfileobj(fi, fo)
    ft = fasttext.load_model(str(FT_BIN))
    ft_vec = lambda tok: ft.get_word_vector(tok.replace('▁', ''))

    dec_pt, stats = sdv.build_pertoken_decoder(
        vi_emb_full, neo_dec, target_vocab, new_to_old, src_vocab,
        anchor_map, specials, ft_vec, device=DEVICE, min_anchors=8)
    print(stats)
    name = f'{VARIANT_PREFIX}_decpertoken'
    sdv.write_variant(V5_DIR, arm_dir(name), dec_pt * DECODER_WEIGHT_SCALE,
                      {'init_name': name, 'decoder_init': 'projected_from_neobert_decoder',
                       'decoder_weight_scale': DECODER_WEIGHT_SCALE})
    ARMS_BUILT.append(('decpertoken', PROJECT_ROOT / 'init' / name))
else:
    print('BUILD_PERTOKEN=False — skipped')

In [ ]:
# ── Health gate: load every built artifact + the v5 reference, check finite forward + step-0 MLM loss ──
import importlib as _il
from transformers import AutoModelForMaskedLM

for c in glob.glob('/root/.cache/huggingface/modules/transformers_modules/*'):
    if Path(c).is_dir(): shutil.rmtree(c, ignore_errors=True)
for k in [k for k in sys.modules if k.startswith('transformers_modules')]: del sys.modules[k]
_il.invalidate_caches()

SENTS = ['Việt Nam là một quốc gia ở Đông Nam Á.',
         'Hôm nay thời tiết rất đẹp và trời trong xanh.',
         'Kinh tế Việt Nam tăng trưởng trong năm qua.',
         'Trẻ em cần được tiêm phòng đầy đủ để tránh bệnh.']

@torch.no_grad()
def mlm_check(model_dir):
    m = AutoModelForMaskedLM.from_pretrained(model_dir, trust_remote_code=True).to(DEVICE).eval()
    t = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    enc = t(SENTS, padding=True, truncation=True, max_length=32, return_tensors='pt').to(DEVICE)
    ids = enc['input_ids']; torch.manual_seed(0)
    pm = torch.full(ids.shape, 0.2)
    for sid in set(t.all_special_ids): pm[ids.cpu() == sid] = 0
    msk = torch.bernoulli(pm).bool().to(DEVICE)
    lab = torch.full_like(ids, -100); lab[msk] = ids[msk]
    mids = ids.clone(); mids[msk] = t.mask_token_id
    wfin = all(bool(torch.isfinite(p).all()) for p in m.parameters())
    lg = m(input_ids=mids, attention_mask=enc['attention_mask']).logits
    fin = bool(torch.isfinite(lg).all())
    loss = F.cross_entropy(lg.reshape(-1, lg.size(-1)).float(), lab.reshape(-1), ignore_index=-100).item()
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return fin, wfin, loss

rows = [('A v5 (reference)', V5_MODEL)] + [(lbl, d / 'model') for lbl, d in ARMS_BUILT]
print(f"{'arm':22s} {'artifact':46s} finite wfin   loss")
allok = True
for lbl, d in rows:
    fin, wfin, loss = mlm_check(d)
    allok &= fin
    print(f'{lbl:22s} {d.parent.name:46s} {str(fin):6s} {str(wfin):5s} {loss:8.3f}')
print('=' * 92)
print('ALL ARMS FINITE — train each in nb02 with MODE=new, fresh RUN_NAME, matched steps/seed.'
      if allok else 'SOME ARM NON-FINITE — inspect before training.')
print('\nnb02 BASE_MODEL_REF for each arm:')
for lbl, d in rows:
    print(f"  {lbl:22s} 'init/{d.parent.name}/model'")